In [2]:
import sys
print(sys.version)

3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]


In [3]:
!pip install syncode

Defaulting to user installation because normal site-packages is not writeable


In [1]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.14.1
Uninstalling torch-2.14.1:
  Successfully uninstalled torch-2.14.1
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 3.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 99.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 107.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 70.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 108.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 20.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 143.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 116.1 M

In [2]:
#restart kernel
import torch
print(torch.__version__)
print(torch.version.cuda)

2.5.1+cu121
12.1


In [3]:
import torch
from syncode import SyncodeLogitsProcessor
from syncode import Grammar
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda'
model_name = "microsoft/phi-2"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/home/nkhoang2/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.39it/s]


In [7]:
grammar_str = r""" start: FLOAT " " OP " " FLOAT "\n"
FLOAT: /-?[0-9]+\.[0-9]+/
OP: "+" | "-" | "*" | "/"
"""
extract_grammar = Grammar(grammar_str)
syncode_logits_processor = SyncodeLogitsProcessor(
    grammar=extract_grammar, tokenizer=tokenizer, parse_output_only=True
)

[2026-10-25 10:54:56,580-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_mask_3558915015_50257.pkl exist: False
[2026-10-25 10:54:56,581-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /shared/storage-01/users/nkhoang2/cs521_hw3/cache/mask_stores/CodeGenTokenizerFast/grammar_mask_3558915015_50257.pkl.
[2026-10-25 10:54:56,974-syncode.mask_store.fsm_set] - 4 FSMs with 16 states initialized in 0.00 seconds
[2026-10-25 10:54:56,974-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-25 10:54:56,985-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 16 to 5


100%|██████████| 16/16 [00:03<00:00,  4.60it/s]


[2026-10-25 10:55:00,517-syncode.mask_store.mask_store] - Time taken to create mask store: 3.93 seconds


In [46]:
prompt_str = """ Extract the two operands (as floats) and the operator from the natural language query.
MUST output in format: <operand1> <operator> <operand2>. MUST NOT perform the calculation or generate any other text.

Query: What is 327. multiplied by 11.0?
Output: 327.0 * 11.0
Query: What is 2 plus 3?
Output: 2.0 + 3.0
Query: 5 minus 2 is equal to?
Output: 5.0 - 2.0
Query: Without giving any other middle step, give the answer to 3 divided by 3/2.
Output: 3.0 / 1.5
"""

In [48]:
OPS = {
    "+": lambda a, b: a + b,
    "-": lambda a, b: a - b,
    "*": lambda a, b: a * b,
    "/": lambda a, b: a / b,
}

def run_syncode(textual_query):
    input_prompt = prompt_str + f"Query: {textual_query}\nOutput:"
    syncode_logits_processor.reset()   # must reset before every generation
    inputs = tokenizer(input_prompt, return_tensors="pt").input_ids.to(device)
    
    output = model.generate(
        inputs,
        max_new_tokens=64,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False, #greedy
        logits_processor=[syncode_logits_processor]
    )
    out = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    try:
        a, op, b = out.split()
        a, b = float(a), float(b)
        return [a, b], op, float(OPS[op](a, b))
    except Exception as e:
        print(f"Error processing query: {textual_query}, output: {out}")
        return None, None, None


In [51]:
def check(q, exp_ops, exp_op, exp_ans):
    ops, op, ans = run_syncode(q)
    assert ops == exp_ops, f"{q}: operands {ops} != {exp_ops}"
    assert op == exp_op, f"{q}: operator {op} != {exp_op}"
    assert abs(ans - exp_ans) < 1e-6, f"{q}: answer {ans} != {exp_ans}"

check("What is 327. multiplied by 11.0?", [327.0, 11.0], "*", 3597.0)
check("45.1 plus 23.54",         [45.1, 23.54], "+", 68.64)
check("Give the answer to 120.4 divided by 4.0?",    [120.4, 4.0],  "/", 30.1)
check("Do not give me any other operands or middle step. Just a single integer final answer. What's 5 minus 11?", [5.0, 11.0], "-", -6.0)

print("All tests passed")


All tests passed
